1. # Practical 2 report: from relationships to predictive models

**Student:** Valeriia  
**Date:** [date]  
**Practical:** 2 - Regression and evaluation


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import dcor
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml, load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 240926

In [ ]:
diabetes = load_diabetes(as_frame=True).frame

# The age feature is standardized; qcut creates relative low/middle/high groups.
diabetes["sex"] = pd.Categorical(
    np.where(diabetes["sex"] > 0, "sex_1", "sex_2"),
    categories=["sex_1", "sex_2"],
)
diabetes["age_group"] = pd.qcut(
    diabetes["age"], q=3, labels=["young", "adult", "senior"]
)

diabetes.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,age_group
0,0.038076,sex_1,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,senior
1,-0.001882,sex_2,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,adult
2,0.085299,sex_1,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,senior
3,-0.089063,sex_2,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,young
4,0.005383,sex_2,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,adult


In [25]:
diabetes.shape

(442, 12)

In [24]:
diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   age        442 non-null    float64 
 1   sex        442 non-null    category
 2   bmi        442 non-null    float64 
 3   bp         442 non-null    float64 
 4   s1         442 non-null    float64 
 5   s2         442 non-null    float64 
 6   s3         442 non-null    float64 
 7   s4         442 non-null    float64 
 8   s5         442 non-null    float64 
 9   s6         442 non-null    float64 
 10  target     442 non-null    float64 
 11  age_group  442 non-null    category
dtypes: category(2), float64(10)
memory usage: 35.8 KB


## 2. Data-preparation readiness

Complete the checklist before describing modelling results. Tick an item only after checking the data or documenting a justified decision.

### Scientific purpose and provenance

- [X] Is the biological question and prediction task stated clearly? To predict how physiological parameters impact targed score aka disease progression.
- [X] Is the unit of observation clear? All the measurments are normalized, so they don't have units. Age and sex were converted manually to categorical types.
- [X] Do I know where the data came from, how they were generated, and under which access conditions they may be used? From the sklearn library.
- [X] Have I recorded the data version, processing history, and transformations already applied? They are in the sklearn documentation.

### Table shape and identifiers

- [X] Is the table tidy: one observation per row, one variable per column, and one value per cell? Yes.
- [X] If needed, have I transposed the original assay into the format expected by the analysis tool? Was already done.
- [X] Are row names, column names, and identifiers clear, unique, and stable? Yes.
- [ ] Can every feature and observation be traced to its source? **???**
- [ ] Are the feature matrix and metadata aligned by an explicit identifier rather than row order? **??**

In [28]:
print("Shape:", diabetes.shape)
print("Column names are unique:", diabetes.columns.is_unique)

Shape: (442, 12)
Column names are unique: True


In [29]:
numeric_columns = diabetes.select_dtypes(include="number").columns.tolist()
categorical_columns = diabetes.select_dtypes(exclude="number").columns.tolist()
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)
print("Constant columns:", [column for column in diabetes.columns if diabetes[column].nunique(dropna=False) <= 1])

Numeric columns: ['age', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6', 'target']
Categorical columns: ['sex', 'age_group']
Constant columns: []


### Types, values, and measurement meaning

- [X] Are numerical, categorical, ordinal, text, date, and identifier columns correctly classified? Yes.
- [X] Are units, scales, encodings, reference categories, and detection limits understood? Yes
- [X] Have impossible values, inconsistent labels, unexpected categories, constant features, and extreme values been checked? No.
- [ ] Are normalization, scaling, log transformation, and batch-correction steps documented? They should be in the sklearn documentation. **???**

In [31]:
meta_cols = ["sex", "age_group"]
measured_cols = [c for c in diabetes.columns if c not in meta_cols]
X_all = diabetes[measured_cols]

print("Infinite values:", int(np.isinf(X_all.to_numpy(dtype=float)).sum()),
      "| negative values:", int((X_all < 0).sum().sum()),
      "| zeros:", int((X_all == 0).sum().sum()),
      "| constant columns:", int((X_all.nunique() <= 1).sum()))
stacked = X_all.stack()
print("Negative value(s) [(row, protein): value]:", stacked[stacked < 0].round(4).to_dict())
print("Value range:", round(float(X_all.min().min()), 3), "to", round(float(X_all.max().max()), 3))
display(X_all.describe().T[["min", "50%", "max"]].sort_values("max", ascending=False).head(5).round(3))

Infinite values: 0 | negative values: 2157 | zeros: 0 | constant columns: 0
Negative value(s) [(row, protein): value]: {(0, 's1'): -0.0442, (0, 's2'): -0.0348, (0, 's3'): -0.0434, (0, 's4'): -0.0026, (0, 's6'): -0.0176, (1, 'age'): -0.0019, (1, 'bmi'): -0.0515, (1, 'bp'): -0.0263, (1, 's1'): -0.0084, (1, 's2'): -0.0192, (1, 's4'): -0.0395, (1, 's5'): -0.0683, (1, 's6'): -0.0922, (2, 'bp'): -0.0057, (2, 's1'): -0.0456, (2, 's2'): -0.0342, (2, 's3'): -0.0324, (2, 's4'): -0.0026, (2, 's6'): -0.0259, (3, 'age'): -0.0891, (3, 'bmi'): -0.0116, (3, 'bp'): -0.0367, (3, 's3'): -0.036, (3, 's6'): -0.0094, (4, 'bmi'): -0.0364, (4, 's4'): -0.0026, (4, 's5'): -0.032, (4, 's6'): -0.0466, (5, 'age'): -0.0927, (5, 'bmi'): -0.0407, (5, 'bp'): -0.0194, (5, 's1'): -0.069, (5, 's2'): -0.0793, (5, 's4'): -0.0764, (5, 's5'): -0.0412, (5, 's6'): -0.0963, (6, 'age'): -0.0455, (6, 'bmi'): -0.0472, (6, 'bp'): -0.016, (6, 's1'): -0.0401, (6, 's2'): -0.0248, (6, 's4'): -0.0395, (6, 's5'): -0.0629, (6, 's6'): -0.0

,min,50%,max
target,25.000,140.500,346.000
s2,-0.116,-0.004,0.199
s4,-0.076,-0.003,0.185
s3,-0.102,-0.007,0.181
bmi,-0.090,-0.007,0.171


In [35]:
print("Categorical levels:", {c: sorted(diabetes[c].unique()) for c in meta_cols})

Categorical levels: {'sex': ['sex_1', 'sex_2'], 'age_group': ['adult', 'senior', 'young']}


### Missing values

- [X] Have missing values been detected using all representations, not only `NaN`? No missing values.
- [X] Have missing values been measured by row, column, and relevant group? No missing values.
- [X] Do I know whether missingness is structural, technical, or plausibly related to biology or the target? No missing values.
- [X] Have retention, removal, or imputation been justified? No missing values.
- [X] If imputing, will the imputer be fitted on training data only? No missing values.

In [36]:
missing_by_column = X_all.isna().sum().sort_values(ascending=False)
missing_by_row = X_all.isna().sum(axis=1)

display(missing_by_column.to_frame("missing_count"))
display((missing_by_column / len(diabetes)).to_frame("missing_fraction"))
print("Rows with at least one missing value:", int((missing_by_row > 0).sum()))
print("Rows with more than one missing value:", int((missing_by_row > 1).sum()))

,missing_count
age,0
bmi,0
bp,0
s1,0
s2,0
s3,0
s4,0
s5,0
s6,0
target,0


,missing_fraction
age,0.0
bmi,0.0
bp,0.0
s1,0.0
s2,0.0
s3,0.0
s4,0.0
s5,0.0
s6,0.0
target,0.0


Rows with at least one missing value: 0
Rows with more than one missing value: 0


### Replicates, dependence, and grouping

- [X] Have exact and near-duplicate observations been checked? No replicates.
- [X] Are observations biological replicates, technical replicates, repeated measurements, pseudo-replicates, or independent samples? No replicates.
- [ ] Is the independent biological unit identified by a grouping variable? Yes **???**
- [X] Will related observations remain in the same train/validation/test split? No, the count of samples distributed across all samples regularly.
- [ ] Is the provenance of synthetic or augmented observations recorded? No **???**

In [42]:
print("Exact duplicate rows", int(diabetes.duplicated().sum()))

for column in ["age_group", "sex"]:
    print(f"\nCounts for {column}:")
    display(diabetes[column].value_counts(dropna=False).to_frame("count"))

Exact duplicate rows 0

Counts for age_group:


,count
age_group,
adult,152
young,147
senior,143



Counts for sex:


,count
sex,
sex_2,235
sex_1,207


In [43]:
print("\nCounts by age and sex:")
display(diabetes.groupby(["age_group", "sex"], observed=False).size().rename("count").to_frame())


Counts by age and sex:


count
age_group sex         
young     sex_1     57
          sex_2     90
adult     sex_1     63
          sex_2     89
senior    sex_1     87
          sex_2     56

### Target, leakage, confounding, and reproducibility

- [X] Is the target defined before modelling? Yes.
- [ ] Have identifiers, post-outcome variables, duplicated target information, and other leakage sources been excluded? Idk **???**
- [ ] Have batch, site, subject, treatment, sex, age, time, and other possible confounders been considered? Yes ( like I had just to find that there are some kinds of possible confounders?) **???**
- [ ] Does the planned split represent the biological generalization question? **???**
- [X] Are random seeds, software versions, file names, shapes, and preprocessing decisions recorded? Yes, in this notebook.

**Preparation summary:**

- Biological unit: `unitless, normalized data`
- Number of observations and features: `442, 12`
- Target and feature definition: `Target - numerical disease progression score, featurs - physiologocsl measured parameters`
- Missing-value decision: `no missing values`
- Replicate/grouping decision: `no replicates`
- Main confounder or leakage source: **???**
- Train/test split and justification: `80/20 randombecause there are no replicated measurments` **???**
- Remaining concerns: `none`

In [44]:
TARGET = "target"

for grouping_variable in ["age_group", "sex"]:
    print(f"\nTarget summary by {grouping_variable}:")
    display(diabetes.groupby(grouping_variable, observed=False)[TARGET].agg(["count", "mean", "std"]))


Target summary by age_group:


,count,mean,std
age_group,,,
young,147,135.639456,71.898325
adult,152,151.526316,81.634377
senior,143,169.734266,73.907246



Target summary by sex:


,count,mean,std
sex,,,
sex_1,207,155.666667,78.453313
sex_2,235,149.021277,75.905781


## 3 Your  task: Diabetes regression

### 3.1 Data presentation

Identify:

- one observation;
- the target variable;
- predictor variables;
- dimensionality of the dataset;
- whether the predictors and target have physical units or transformed scales.

### 3.2 Descriptive analysis

- Summarize the target.
- Visualize its distribution.
- Visualize at least three predictor-target relationships.
- State the biological or predictive question addressed by each plot.

### 3.3 Correlation

- Calculate correlation using ar least 2 different methods.
- Identify the predictor most strongly correlated with the target.
- Compare results.
- Explain why correlation is not sufficient for feature selection or causal interpretation.

### 4.4 Simple regression

Choose one predictor and fit a simple linear regression. Report and interpret:

- intercept;
- slope;
- the scale of the predictor;
- whether the slope is a causal effect.

### 4.5 Residual diagnostics

Produce:

- observed versus predicted plot;
- residuals versus fitted plot;
- residual histogram;
- Q-Q plot.

Discuss linearity, homoscedasticity, residual normality, unusual observations, and any limitations of the diagnostics.

### 4.6 Multiple regression and multicollinearity

Fit a model using all available predictors. Inspect:

- coefficients;
- predictor correlations;
- possible multicollinearity;
- whether standardized predictors change coefficient interpretation.

### 4.7 Train/test evaluation

Create a justified train/test split and report MAE, MSE, RMSE, and $R^2$. Compare simple and multiple regression on test data.

Explain whether the split answers the intended biological or clinical generalization question.

### 4.8 Independent-task conclusion

Answer:

1. Which model generalized better?
2. Did adding more variables help?
3. Are any regression assumptions questionable?
4. Are coefficient interpretations straightforward?
5. What is one important limitation of the dataset or model?

Remember that the predictors from `load_diabetes()` are already standardized.